## SHAP(Shapley Additive Explanations)

- `SHAP` can be used to understand how our machine learning models work.

- It can also be used how the model feature has contributed to a prediction.

- It can also be used to understand what trends the model is using to make the predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

import shap
shap.initjs()

In [ ]:
# Load the dataset
data=pd.read_csv("data/abalone.data",
                 names=['sex', 'length', 'diameter', 'height',
                        'whole weight', 'shucked weight',
                        'viscera weight', 'shell weight', 'rings'])

In [ ]:
data.head()

In [ ]:
data.shape

### Visualizing the Dataset

In [ ]:
plt.scatter(data['whole weight'], data['rings'])
plt.xlabel('Whole weight', size=12)
plt.ylabel('Number of rings', size=12)

In [ ]:
plt.boxplot(data[data.sex=='I']['rings'], positions=[1])
plt.boxplot(data[data.sex=='M']['rings'], positions=[2])
plt.boxplot(data[data.sex=='F']['rings'], positions=[3])

plt.xticks(ticks=[1, 2, 3], labels=['I', 'M', 'F'], size=12)
plt.xlabel('Sex', size=15)
plt.ylabel('Number of rings', size=15)

In [ ]:
cont_features=['length', 'diameter', 'height', 
               'whole weight', 'shucked weight',
               'viscera weight', 'shell weight', 'rings']

corr_matrix=pd.DataFrame(data[cont_features], columns=cont_features).corr()

sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    center=0,
    annot=True,
    fmt='.1g'
)

In [ ]:
x=data[['sex', 'length', 'height', 'shucked weight',
        'viscera weight', 'shell weight']]
y=data['rings']

# Create dummy variables
x['sex.M']=[1 if s=='M' else 0 for s in x['sex']]
x['sex.F']=[1 if s=='F' else 0 for s in x['sex']]
x['sex.I']=[1 if s=='I' else 0 for s in x['sex']]

x=x.drop('sex', axis=1)

x.head(10)

### Modelling

In [ ]:
model=xgb.XGBRegressor(objective='reg:squarederror')

In [ ]:
model.fit(x, y)

In [ ]:
y_pred=model.predict(x)

plt.figure(figsize=(5, 5))

plt.scatter(y, y_pred)
plt.plot(
    [0, 30],
    [0, 30],
    color='r',
    linestyle='-',
    linewidth=2
)

plt.ylabel('Predicted', size=12)
plt.xlabel('Actual', size=12)

### Standard SHAP Values

In [ ]:
explainer=shap.Explainer(model=model)
shap_values=explainer(x)

In [ ]:
print(shap_values.values)

In [ ]:
np.shape(shap_values.values)

### Waterfall Plot

In [ ]:
shap.plots.waterfall(shap_values[0])

- `E[f(X)]`: Average predicted number of rings across the 4177 samples present in the dataset.

- `f(X)`: Predicted number of rings for that particular sample.

- The SHAP values tell us how each model feature has contributed to the difference between the prediction and the average prediction.

### Force Plot

In [ ]:
shap.plots.force(shap_values[0])

### Stacked Force Plot

In [ ]:
shap.plots.force(shap_values[0:100])

### Absolute Mean SHAP Plot

In [ ]:
shap.plots.bar(shap_values)

- The `absolute mean` shap plot tells us which feature is more important for the model prediction.

- The features that have made large positive or negative contributions will have large mean SHAP value i.e; these are the features that have made significant contributions to the model's predictions.

### Beeswarm Plot

In [ ]:
shap.plots.beeswarm(shap_values)

- The `beeswarm` plot can be used to highlight the important relationships.

### Dependence Plots

In [ ]:
shap.plots.scatter(shap_values[:, 'shell weight'])

In [ ]:
shap.plots.scatter(shap_values[:, 'shucked weight'])

In [ ]:
shap.plots.scatter(shap_values[:, 'shell weight'],
                   color=shap_values[:, 'shucked weight'])

### Violin Plot

In [ ]:
shap.summary_plot(shap_values.values, x, plot_type='violin')

### Layered Violin Plot

In [ ]:
shap.plots.violin(shap_values)

In [ ]:
shap.plots.violin(shap_values, plot_type='layered_violin')

### HeatMap Plot

In [ ]:
shap.plots.heatmap(shap_values)

- We focus on patterns between shap values and groups of instances.

- By default, the instances are ordered using a hierarchial clustering algorithm but choosing our own ordering is more useful for finding patterns.

## Binary Target Variable

In [ ]:
y_bin=[1 if y_>10 else 0 for y_ in y]

In [ ]:
model_bin=xgb.XGBClassifier(objective='binary:logistic')

In [ ]:
model_bin.fit(x, y_bin)

In [ ]:
explainer_bin=shap.Explainer(model=model_bin)
shap_values_bin=explainer_bin(x)

In [ ]:
print(shap_values_bin.shape)

In [ ]:
shap.plots.waterfall(shap_values_bin[0])

- `E[f(X)]`: average predicted log odds across all the abalone.

- We can say that `shucked weight` has increased the probability that the model will predict an above average number of rings.

- Negative shap values will decrease the log odds.

In [ ]:
shap.plots.force(shap_values_bin[0], link='logit')

In [ ]:
shap.plots.bar(shap_values_bin)

## Multi-Class Target Variable

In [ ]:
y_cat=[2 if y_>12 else 1 if y_>8 else 0 for y_ in y]

In [ ]:
model_cat=xgb.XGBClassifier(objective='binary:logistic')

In [ ]:
model_cat.fit(x, y_cat)

In [ ]:
explainer_cat=shap.Explainer(model=model_cat)
shap_values_cat=explainer_cat(x)

In [ ]:
print(shap_values_cat.shape)

In [ ]:
model_cat.predict_proba(X=x)[0]

In [ ]:
shap.plots.waterfall(shap_values_cat[0, :, 0])

In [ ]:
shap.plots.waterfall(shap_values_cat[0, :, 1])

In [ ]:
shap.plots.waterfall(shap_values_cat[0, :, 2])

In [ ]:
def softmax(x):
    e_x=np.exp(x-np.max(x))
    return e_x/e_x.sum(axis=0)

temp=[0.383, -0.106, 1.211]
softmax(x=temp)

### Aggregated SHAP

In [ ]:
# Calculate the mean SHAP values for each class
mean_0=np.mean(np.abs(shap_values_cat.values[:, :, 0]), axis=0)
mean_1=np.mean(np.abs(shap_values_cat.values[:, :, 1]), axis=0)
mean_2=np.mean(np.abs(shap_values_cat.values[:, :, 2]), axis=0)

df=pd.DataFrame({'young': mean_0, 'medium': mean_1, 'old': mean_2})

# Plot mean shap values
fig, ax=plt.subplots(1, 1, figsize=(20, 10))
df.plot.bar(ax=ax)

ax.set_ylabel('Mean SHAP', size=20)
ax.set_xticklabels(x.columns, rotation=45, size=20)
ax.legend(fontsize=30)
plt.show()

In [ ]:
preds=model_cat.predict(x)

In [ ]:
new_shap_values=[]

for i, pred in enumerate(preds):
    new_shap_values.append(shap_values_cat.values[i][:, pred])

# Replace the SHAP values
shap_values_cat.values=np.array(new_shap_values)

In [ ]:
shap_values_cat.shape

In [ ]:
shap.plots.bar(shap_values_cat)

In [ ]:
shap.plots.beeswarm(shap_values_cat)

### SHAP Interaction

In [ ]:
explainer_int=shap.Explainer(model=model)
shap_values_int=explainer_int.shap_interaction_values(x)

In [ ]:
shap_values_int.shape

In [ ]:
shap_0=np.round(shap_values_int[0], 2)
shap_0_df=pd.DataFrame(shap_0, index=x.columns, columns=x.columns)

In [ ]:
shap_0_df.head()

In [ ]:
sns.heatmap(shap_0_df, cmap='coolwarm', annot=True)
plt.show()

### Mean SHAP Interaction

In [ ]:
mean_shap=np.abs(shap_values_int).mean(0)
mean_shap=np.round(mean_shap, 1)

mean_shap_df=pd.DataFrame(mean_shap, index=x.columns, columns=x.columns)

mean_shap_df.where(
    mean_shap_df.values==np.diagonal(mean_shap_df),
    mean_shap_df.values*2,
    inplace=True
)

sns.set(font_scale=1)
sns.heatmap(mean_shap_df, cmap='coolwarm', annot=True)
plt.yticks(rotation=0)
plt.show()